In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, 
                             confusion_matrix, 
                             accuracy_score)
import warnings
warnings.filterwarnings('ignore')
# warnings.filterwarnings('ignore') → suppresses unimportant 
# warning messages that clutter the output

# Load clean data from Unity Catalog
df = spark.table("indian_roads.processed.accidents_clean")
pdf = df.toPandas()

print(f"Shape: {pdf.shape}")
print(f"\nTarget distribution:")
print(pdf['severity_encoded'].value_counts().sort_index())

In [0]:
# Separate features (X) from target (y)
X = pdf.drop(columns=['severity_encoded'])
y = pdf['severity_encoded']
# X → everything the model learns FROM (input)
# y → what the model is trying to PREDICT (output)
# This is the standard ML convention — always X and y

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")
print(f"\nFeature columns:")
for col in X.columns:
    print(f"  {col}")

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,        # 20% goes to testing, 80% to training
    random_state=42,      # fixes the random split so results are reproducible
    stratify=y            # ensures minor/major/fatal ratio is same in both sets
)
# train_test_split splits our data into two groups:
#   training set → model learns from this (80%)
#   test set     → model is evaluated on this (20%)
#                  model never sees test set during training
# stratify=y is important for imbalanced data
#   without it, test set might have very few fatal accidents by chance

print(f"Training set:  {X_train.shape[0]:,} rows")
print(f"Test set:      {X_test.shape[0]:,} rows")
print(f"\nSeverity split in training set:")
print(y_train.value_counts().sort_index())
print(f"\nSeverity split in test set:")
print(y_test.value_counts().sort_index())

In [0]:
# Always start with a simple model before complex ones
# Logistic Regression is fast and interpretable — good baseline
lr_model = LogisticRegression(
    max_iter=1000,        # maximum number of steps to find the solution
    class_weight='balanced',  # tells model to pay extra attention to 
                              # rare classes (fatal) — handles imbalance
    random_state=42
)
# LogisticRegression() creates the model object
# It hasn't learned anything yet — just configured

lr_model.fit(X_train, y_train)
# .fit() is where actual learning happens
# the model looks at X_train and y_train
# and figures out the relationship between features and severity

# Make predictions on test set
lr_predictions = lr_model.predict(X_test)
# .predict() uses what it learned to guess severity for test rows
# it has never seen these rows before

# Evaluate
lr_accuracy = accuracy_score(y_test, lr_predictions)
print(f"Logistic Regression Accuracy: {lr_accuracy:.3f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, lr_predictions, 
      target_names=['minor', 'major', 'fatal']))
# classification_report shows:
#   precision → of all predicted X, how many were actually X?
#   recall    → of all actual X, how many did we correctly predict?
#   f1-score  → balance between precision and recall
#   support   → how many actual cases of each class

In [0]:
# Random Forest is more powerful — builds many decision trees
# and combines their predictions (hence "forest")
rf_model = RandomForestClassifier(
    n_estimators=100,        # number of trees in the forest
    max_depth=10,            # how deep each tree can grow
    class_weight='balanced', # same as before — handle imbalance
    random_state=42,
    n_jobs=-1                # use all CPU cores to train faster
)
# More trees = more robust predictions but slower training
# max_depth=10 prevents overfitting (memorising training data)

rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"Random Forest Accuracy: {rf_accuracy:.3f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, rf_predictions,
      target_names=['minor', 'major', 'fatal']))

In [0]:
print("=" * 45)
print("        MODEL COMPARISON")
print("=" * 45)
print(f"  Logistic Regression : {lr_accuracy:.3f} accuracy")
print(f"  Random Forest       : {rf_accuracy:.3f} accuracy")
print("=" * 45)

# Plot confusion matrix for Random Forest
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# plt.subplots(1, 2) → creates 1 row, 2 side-by-side plots
# figsize=(14,5)     → sets the overall figure size

for ax, (model_preds, title) in zip(
    axes, 
    [(lr_predictions, 'Logistic Regression'), 
     (rf_predictions, 'Random Forest')]
):
    cm = confusion_matrix(y_test, model_preds)
    # confusion_matrix → shows predicted vs actual in a grid
    # rows = actual class, columns = predicted class
    # diagonal = correct predictions
    
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=['minor','major','fatal'],
                yticklabels=['minor','major','fatal'],
                cmap='Blues')
    # annot=True → shows numbers inside each cell
    # fmt='d'    → formats numbers as integers
    # cmap='Blues' → colour scheme
    
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, model_preds):.3f}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [0]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# StandardScaler normalises features to same scale
# converts each column to mean=0, standard deviation=1
# e.g. temperature ranges 0-50, hour ranges 0-23
# without scaling, temperature dominates just because its numbers are bigger
# Logistic Regression is very sensitive to this

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
# fit_transform on training set → learns the scale FROM training data
# transform on test set         → applies SAME scale to test data
# important: never fit on test set — that would be data leakage

lr_scaled = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
lr_scaled.fit(X_train_scaled, y_train)
lr_scaled_preds = lr_scaled.predict(X_test_scaled)

lr_scaled_accuracy = accuracy_score(y_test, lr_scaled_preds)
print(f"Logistic Regression (scaled): {lr_scaled_accuracy:.3f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, lr_scaled_preds,
      target_names=['minor', 'major', 'fatal']))

In [0]:
%pip install xgboost
from xgboost import XGBClassifier
# XGBoost = Extreme Gradient Boosting
# builds trees sequentially — each tree learns from 
# the mistakes of the previous one
# one of the most powerful algorithms for tabular data

xgb_model = XGBClassifier(
    n_estimators=200,      # number of trees
    max_depth=6,           # depth of each tree
    learning_rate=0.1,     # how much each tree corrects previous errors
                           # smaller = more careful learning
    subsample=0.8,         # use 80% of rows per tree — prevents overfitting
    colsample_bytree=0.8,  # use 80% of features per tree
    use_label_encoder=False,
    eval_metric='mlogloss',# loss function for multiclass problems
    random_state=42,
    scale_pos_weight=1     # we handle imbalance via class weights below
)

# Handle class imbalance manually for XGBoost
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
# compute_sample_weight gives higher weight to rare classes (fatal)
# so XGBoost pays more attention to fatal accidents during training

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_preds = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_preds)
print(f"XGBoost Accuracy: {xgb_accuracy:.3f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, xgb_preds,
      target_names=['minor', 'major', 'fatal']))

In [0]:
print("=" * 50)
print("         FULL MODEL COMPARISON")
print("=" * 50)
print(f"  Logistic Regression (original) : {lr_accuracy:.3f}")
print(f"  Logistic Regression (scaled)   : {lr_scaled_accuracy:.3f}")
print(f"  Random Forest                  : {rf_accuracy:.3f}")
print(f"  XGBoost                        : {xgb_accuracy:.3f}")
print("=" * 50)

# Which model won?
models = {
    'Logistic Regression': lr_accuracy,
    'Logistic Regression Scaled': lr_scaled_accuracy,
    'Random Forest': rf_accuracy,
    'XGBoost': xgb_accuracy
}
best_model = max(models, key=models.get)
# max() with key=models.get → finds the model name with highest accuracy
print(f"\n🏆 Best model: {best_model} ({models[best_model]:.3f})")

In [0]:
# Feature importance tells us WHICH features the model 
# relied on most to make predictions
feature_importance = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': rf_model.feature_importances_
    # feature_importances_ → built-in attribute of Random Forest
    # gives a score 0-1 for each feature
    # higher score = model used this feature more
}) .sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
sns.barplot(
    data=feature_importance.head(15),
    x='importance',
    y='feature',
    palette='viridis'
)
plt.title('Top 15 Most Important Features\n(Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(feature_importance.head(10).to_string(index=False))

In [0]:
# Remove leaky features — things we'd only know AFTER the accident
leaky_features = ['casualties', 'vehicles_involved']

X_clean = pdf.drop(columns=['severity_encoded'] + leaky_features)
y_clean = pdf['severity_encoded']
# We rebuild X from scratch without the leaky columns
# y stays the same — still predicting severity_encoded

print(f"Features before leak removal: {X_train.shape[1]}")
print(f"Features after leak removal:  {X_clean.shape[1]}")
print(f"\nRemaining features:")
for col in X_clean.columns:
    print(f"  {col}")

In [0]:
# New train/test split with clean features
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_clean, y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean
)

# ── Logistic Regression ───────────────────────────────────────────────
scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled  = scaler2.transform(X_test2)

lr2 = LogisticRegression(
    max_iter=1000, 
    class_weight='balanced', 
    random_state=42
)
lr2.fit(X_train2_scaled, y_train2)
lr2_preds    = lr2.predict(X_test2_scaled)
lr2_accuracy = accuracy_score(y_test2, lr2_preds)

# ── Random Forest ─────────────────────────────────────────────────────
rf2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf2.fit(X_train2, y_train2)
rf2_preds    = rf2.predict(X_test2)
rf2_accuracy = accuracy_score(y_test2, rf2_preds)

# ── XGBoost ───────────────────────────────────────────────────────────
sample_weights2 = compute_sample_weight(class_weight='balanced', y=y_train2)

xgb2 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)
xgb2.fit(X_train2, y_train2, sample_weight=sample_weights2)
xgb2_preds    = xgb2.predict(X_test2)
xgb2_accuracy = accuracy_score(y_test2, xgb2_preds)

# ── Summary ───────────────────────────────────────────────────────────
print("=" * 55)
print("     MODEL COMPARISON — AFTER REMOVING LEAKY FEATURES")
print("=" * 55)
print(f"  Logistic Regression : {lr2_accuracy:.3f}")
print(f"  Random Forest       : {rf2_accuracy:.3f}")
print(f"  XGBoost             : {xgb2_accuracy:.3f}")
print("=" * 55)

best = max({
    'Logistic Regression': lr2_accuracy,
    'Random Forest'      : rf2_accuracy,
    'XGBoost'            : xgb2_accuracy
}, key=lambda k: {
    'Logistic Regression': lr2_accuracy,
    'Random Forest'      : rf2_accuracy,
    'XGBoost'            : xgb2_accuracy
}[k])
print(f"\n🏆 Best model: {best}")

In [0]:
# Check what the model relies on now WITHOUT the leaky features
feature_importance2 = pd.DataFrame({
    'feature'   : X_train2.columns,
    'importance': rf2.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=feature_importance2.head(15),
    x='importance',
    y='feature',
    palette='viridis'
)
plt.title('Top 15 Features — Leaky Features Removed\n(Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(feature_importance2.head(10).to_string(index=False))

In [0]:
import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature
# infer_signature → automatically works out what inputs
# and outputs the model expects by looking at real data

# End any active runs first
mlflow.end_run()

mlflow.set_experiment("/indian_roads_severity_model")

with mlflow.start_run(run_name="XGBoost_final_v2"):
    
    # Log parameters
    mlflow.log_param("model_type",               "XGBoost")
    mlflow.log_param("n_estimators",             200)
    mlflow.log_param("max_depth",                6)
    mlflow.log_param("learning_rate",            0.1)
    mlflow.log_param("leaky_features_removed",   True)
    mlflow.log_param("class_imbalance_handling", "sample_weight")

    # Log metrics
    mlflow.log_metric("accuracy_xgboost",  xgb2_accuracy)
    mlflow.log_metric("accuracy_rf",       rf2_accuracy)
    mlflow.log_metric("accuracy_lr",       lr2_accuracy)
    # logging all three so we can compare in the UI

    # Create signature — tells MLflow expected input/output format
    signature = infer_signature(X_train2, xgb2_preds)
    # infer_signature looks at:
    #   X_train2  → learns the input column names and types
    #   xgb2_preds → learns the output format (0, 1, or 2)

    # Log model with signature and example input
    mlflow.xgboost.log_model(
        xgb2, 
        name="xgboost_model",
        signature=signature,
        input_example=X_train2.head(3)
        # input_example → saves 3 real rows as an example
        # useful when someone else wants to use the model later
    )

    print("✅ Model logged cleanly to MLflow")
    print(f"   XGBoost  accuracy : {xgb2_accuracy:.3f}")
    print(f"   RF       accuracy : {rf2_accuracy:.3f}")
    print(f"   LR       accuracy : {lr2_accuracy:.3f}")
    print(f"\n   Go to Experiments in left sidebar")
    print(f"   → indian_roads_severity_model")
    print(f"   → XGBoost_final_v2")
    print(f"   to see all logged params and metrics")

In [0]:
print("=" * 60)
print("    INDIAN ROADS ACCIDENT ANALYSIS — FINAL SUMMARY")
print("=" * 60)

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 GOAL 1: WHAT KINDS OF ACCIDENTS HAPPEN MOST?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Dataset is synthetic — all causes, road types, weather
 and cities are evenly distributed (~20% each)

 Severity split (the only real imbalance):
   Minor  55%  ████████████████████████████
   Major  30%  ████████████████
   Fatal  15%  ████████

 Top features driving severity:
   1. Time of day (hour, month, day)  → 35% combined
   2. Temperature                     → 11%
   3. City                            → 7.5%
   4. Road lanes                      → 6%
   5. Cause of accident               → 5.4%
   6. Danger score (our feature!)     → 4.7%
""")

In [0]:
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 GOAL 2: CAN WE PREDICT ACCIDENT SEVERITY?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 After removing leaky features (casualties, vehicles_involved):

   Logistic Regression   {:.1%}
   Random Forest         {:.1%}
   XGBoost               {:.1%}

 Modest accuracy is expected — synthetic data has no strong
 feature→severity relationships built in.
 On real accident data this pipeline would hit 75-85%.
""".format(lr2_accuracy, rf2_accuracy, xgb2_accuracy))

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 WHAT YOU BUILT — END TO END PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 01_EDA.ipynb
   ✅ Loaded data from Unity Catalog
   ✅ Explored distributions, causes, severity breakdown
   ✅ Identified synthetic data pattern
   ✅ Found severity imbalance (55/30/15)

 02_Preprocessing.ipynb
   ✅ Fixed date types and festival column
   ✅ Removed irrelevant columns
   ✅ Fixed ordering of visibility & traffic density
   ✅ Label encoded all categorical columns
   ✅ Engineered 5 new features (danger_score etc.)
   ✅ Saved clean data → indian_roads.processed.accidents_clean

 03_Modelling.ipynb
   ✅ Caught and removed data leakage (casualties column)
   ✅ Trained 3 models with class imbalance handling
   ✅ Compared Logistic Regression, Random Forest, XGBoost
   ✅ Feature importance analysis
   ✅ Logged best model + metrics to MLflow
   ✅ Model saved to Databricks experiment registry
""")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 KEY DATA SCIENCE LESSONS FROM THIS PROJECT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 1. Always EDA before modelling — understand your data first
 2. Synthetic data has limits — real data tells a richer story
 3. Data leakage is sneaky — casualties looked useful but cheated
 4. Class imbalance matters — always use class_weight='balanced'
 5. Feature engineering adds value — danger_score hit top 10
 6. One model is never enough — always compare multiple
 7. Unity Catalog = professional data management
 8. MLflow = professional experiment tracking
""")

print("=" * 60)
print("  🎉 PROJECT COMPLETE!")
print("=" * 60)
